# 1. Datenqualität prüfen

In dieser Phase untersuchen wir ausschließlich die Rohdaten. Wir prüfen Aufbau, fehlende Werte, Zielklassen, Wertebereiche und Kategorien. Die Daten werden dabei noch nicht verändert.

In [4]:
import pandas as pd

df = pd.read_csv("data/data.csv")

print("Zeilen und Spalten:", df.shape)
display(df.head())

# pd.read_csv lädt die CSV-Datei in die Tabelle df.
# df.shape zeigt die Anzahl der Zeilen und Spalten.
# df.head() zeigt die ersten fünf Personen.

Zeilen und Spalten: (19719, 23)


,N6,N8,N7,N1,N9,N10,N3,N5,E3,N2,...,E4,age,C4,E1,A4,E10,E9,gender,hand,target
0,1,1,1,1,1,1,2,1,5,5,...,2,53,1,4,5,1,5,Male,Right,Resilient
1,4,2,3,2,2,4,4,3,3,3,...,3,46,2,2,4,5,1,Female,Right,Moderate
2,5,5,5,5,5,5,5,5,1,1,...,4,14,1,5,5,1,5,Female,Right,Moderate
3,5,5,5,5,4,5,4,4,2,4,...,4,19,5,2,4,5,4,Female,Right,Overcontroller
4,3,3,3,3,3,4,3,3,3,3,...,3,25,3,3,5,5,3,Female,Right,Moderate


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19719 entries, 0 to 19718
Data columns (total 23 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   N6      19719 non-null  int64
 1   N8      19719 non-null  int64
 2   N7      19719 non-null  int64
 3   N1      19719 non-null  int64
 4   N9      19719 non-null  int64
 5   N10     19719 non-null  int64
 6   N3      19719 non-null  int64
 7   N5      19719 non-null  int64
 8   E3      19719 non-null  int64
 9   N2      19719 non-null  int64
 10  E5      19719 non-null  int64
 11  N4      19719 non-null  int64
 12  E7      19719 non-null  int64
 13  E4      19719 non-null  int64
 14  age     19719 non-null  int64
 15  C4      19719 non-null  int64
 16  E1      19719 non-null  int64
 17  A4      19719 non-null  int64
 18  E10     19719 non-null  int64
 19  E9      19719 non-null  int64
 20  gender  19695 non-null  str  
 21  hand    19619 non-null  str  
 22  target  19719 non-null  str  
dtypes: int64(20), str(3)
m

In [7]:
# Fehlende Werte in jeder Spalte zählen
df.isna().sum()

N6          0
N8          0
N7          0
N1          0
N9          0
N10         0
N3          0
N5          0
E3          0
N2          0
E5          0
N4          0
E7          0
E4          0
age         0
C4          0
E1          0
A4          0
E10         0
E9          0
gender     24
hand      100
target      0
dtype: int64

### Erste Datenprüfung

- **Umfang:** 19.719 Zeilen und 23 Spalten
- **Fehlende Werte in `gender`:** 24
- **Fehlende Werte in `hand`:** 100
- **Übrige Spalten:** keine fehlenden Werte
- **Nächster Schritt:** Daten weiter untersuchen, bevor wir über die Bereinigung entscheiden

In [8]:
# Zählen, wie oft jeder Persönlichkeitstyp vorkommt
df["target"].value_counts()

target
Moderate           8452
Resilient          6163
Overcontroller     2829
Undercontroller    2275
Name: count, dtype: int64

### Verteilung der Zielklassen

- **Moderate:** 8.452 Personen
- **Resilient:** 6.163 Personen
- **Overcontroller:** 2.829 Personen
- **Undercontroller:** 2.275 Personen
- **Beobachtung:** Die vier Klassen sind unterschiedlich häufig vertreten.*

In [9]:
# Überblick über das Alter: unter anderem kleinster und größter Wert
df["age"].describe()

count    1.971900e+04
mean     5.076703e+04
std      7.121272e+06
min      1.300000e+01
25%      1.800000e+01
50%      2.200000e+01
75%      3.100000e+01
max      1.000000e+09
Name: age, dtype: float64

### Prüfung der Altersspalte

- **Untersucht:** `age` mit `describe()`, um unter anderem kleinsten und größten Wert zu sehen.
- **Mittlerer Wert (Median):** 22 Jahre.
- **Auffälligkeit:** Der größte Wert beträgt 1.000.000.000 Jahre und ist offensichtlich unmöglich.
- **Folge:** Der Durchschnitt ist durch diesen Wert stark verzerrt und beschreibt das typische Alter nicht sinnvoll.
- **Noch offen:** Wie viele auffällige Alterswerte es gibt und wie wir sie bereinigen.

In [17]:
# Alterswerte vom größten zum kleinsten sortieren und die ersten zehn anzeigen
df["age"].sort_values(ascending=False).head(10)

4004     999999999
9987        412434
10715         2000
1649          1999
14628         1998
6614          1998
18661         1998
19614         1998
3016          1997
331           1997
Name: age, dtype: int64

### Größte Alterswerte

- **Prüfung:** Die zehn größten Werte der Spalte `age` wurden angezeigt.
- **Auffälligkeiten:** Die größten Werte sind 999.999.999 und 412.434. Auch Werte zwischen 1997 und 2000 können keine Altersangaben in Jahren sein.
- **Mögliche Erklärung:** Die vierstelligen Werte könnten Geburtsjahre sein; das ist noch nicht bestätigt.
- **Noch offen:** Wie viele Datensätze betroffen sind und wie wir sie bereinigen.

In [18]:
# Zählen, wie viele Altersangaben über 120 liegen
# 120 ist hier nur eine Grenze zur Untersuchung, noch keine Bereinigungsregel
(df["age"] > 120).sum()

np.int64(82)

### Anzahl auffälliger Alterswerte

- **Prüfung:** Altersangaben über 120 Jahre gezählt.
- **Ergebnis:** 82 von 19.719 Datensätzen liegen über dieser Prüfgrenze.
- **Wichtig:** Die Grenze dient bisher nur der Untersuchung; es wurde noch nichts bereinigt.

### Abgleich mit dem Codebook

- **Definition:** `age` gibt das Alter in Jahren an.
- **Ergebnis:** 82 Angaben liegen über 120 Jahren und sind für diese Prüfung auffällig.
- **Folgerung:** Vierstellige Werte dürfen nicht ohne Beleg als Geburtsjahre umgerechnet werden.
- **Bereinigung:** Eine Regel ist noch festzulegen; bisher wurden keine Daten verändert.

In [19]:
# N1: I get stressed out easily.
# Zählen, wie oft jeder Antwortwert bei dieser Frage vorkommt
df["N1"].value_counts()

N1
4    4949
3    4324
5    4308
2    3880
1    2257
0       1
Name: count, dtype: int64

### Antwortverteilung bei N1

- **Frage:** I get stressed out easily.
- **Antwortskala laut Codebook:** 1 = Disagree bis 5 = Agree.
- **Häufigster Wert:** 4 mit 4.949 Antworten.
- **Auffälligkeit:** Der Wert 0 kommt einmal vor und liegt außerhalb der erlaubten Skala.
- **Bereinigung:** Noch keine Daten verändert; wir prüfen zunächst auch die anderen Fragen.

In [20]:
# Alle Frage-Spalten prüfen; age, gender, hand und target sind keine Fragen
# Den kleinsten Antwortwert für jede Frage anzeigen
df.drop(columns=["age", "gender", "hand", "target"]).min()

N6     0
N8     0
N7     0
N1     0
N9     0
N10    0
N3     0
N5     0
E3     0
N2     0
E5     0
N4     0
E7     0
E4     0
C4     0
E1     0
A4     0
E10    0
E9     0
dtype: int64

### Kleinste Werte der Fragebewertungen

- **Erlaubte Skala laut Codebook:** 1 bis 5.
- **Beobachtung:** Bei allen 19 Fragen ist der kleinste vorkommende Wert 0.
- **Einordnung:** 0 liegt außerhalb der erlaubten Skala.
- **Noch offen:** Wie oft 0 je Frage vorkommt und ob dieselben oder verschiedene Datensätze betroffen sind.
- **Bereinigung:** Bisher wurden keine Werte verändert.

In [21]:
# Für jede Frage zählen, wie oft der ungültige Antwortwert 0 vorkommt
# Die vier Nicht-Frage-Spalten werden nur für diese Prüfung ausgelassen
(df.drop(columns=["age", "gender", "hand", "target"]) == 0).sum()

N6     1
N8     1
N7     1
N1     1
N9     1
N10    1
N3     1
N5     1
E3     1
N2     1
E5     1
N4     1
E7     1
E4     1
C4     1
E1     1
A4     1
E10    1
E9     1
dtype: int64

### Häufigkeit ungültiger Fragewerte

- **Prüfung:** Anzahl der `0`-Antworten je Frage gezählt.
- **Ergebnis:** Bei jeder der 19 Fragen kommt `0` genau einmal vor.
- **Noch offen:** Ob alle Nullen zur selben Zeile gehören.
- **Bereinigung:** Noch keine Daten verändert.

In [22]:
# Zählen, wie viele Zeilen bei mindestens einer Frage den Wert 0 haben
# any(axis=1) prüft dafür die Fragewerte innerhalb jeder Zeile
(df.drop(columns=["age", "gender", "hand", "target"]) == 0).any(axis=1).sum()

np.int64(1)

### Datensatz mit ungültigen Fragewerten

- **Ergebnis:** Genau eine Zeile enthält den Wert `0` bei den Fragebewertungen.
- **Schlussfolgerung:** Alle 19 Fragen dieser Zeile haben den ungültigen Wert `0`.
- **Bereinigung:** Diese Zeile muss gesondert behandelt werden; bisher wurde sie nicht verändert.

In [23]:
# Die größten Antwortwerte aller Frage-Spalten anzeigen, damit auch Werte über 5 ausschließen können.
# age, gender, hand und target gehören nicht zu den Fragen
df.drop(columns=["age", "gender", "hand", "target"]).max()

N6     5
N8     5
N7     5
N1     5
N9     5
N10    5
N3     5
N5     5
E3     5
N2     5
E5     5
N4     5
E7     5
E4     5
C4     5
E1     5
A4     5
E10    5
E9     5
dtype: int64

### Größte Werte der Fragebewertungen

- **Erlaubte Skala laut Codebook:** 1 bis 5.
- **Ergebnis:** Der größte Wert ist bei allen 19 Fragen `5`.
- **Einordnung:** Es wurden keine Antwortwerte über der erlaubten Skala gefunden.
- **Bisherige Auffälligkeit:** Eine Zeile enthält bei allen 19 Fragen den ungültigen Wert `0`.

In [24]:
# Die Antwortverteilungen aller 19 Fragen zusammenfassen
# .T dreht die Tabelle: Jede Frage steht dadurch in einer eigenen Zeile
df.drop(columns=["age", "gender", "hand", "target"]).describe().T

,count,mean,std,min,25%,50%,75%,max
N6,19719.0,2.980374,1.320437,0.0,2.0,3.0,4.0,5.0
N8,19719.0,2.803235,1.350648,0.0,2.0,3.0,4.0,5.0
N7,19719.0,3.151935,1.299910,0.0,2.0,3.0,4.0,5.0
N1,19719.0,3.262082,1.308169,0.0,2.0,3.0,4.0,5.0
N9,19719.0,3.135250,1.298573,0.0,2.0,3.0,4.0,5.0
N10,19719.0,2.833764,1.313036,0.0,2.0,3.0,4.0,5.0
N3,19719.0,3.842690,1.138854,0.0,3.0,4.0,5.0,5.0
N5,19719.0,2.951722,1.272889,0.0,2.0,3.0,4.0,5.0
E3,19719.0,3.416755,1.236820,0.0,3.0,4.0,4.0,5.0
N2,19719.0,3.234596,1.177018,0.0,2.0,3.0,4.0,5.0


### Verteilung der Fragebewertungen

- **Umfang:** Für jede der 19 Fragen liegen 19.719 Antworten vor.
- **Mittlere Antwort (Median):** Je nach Frage liegt sie bei 3 oder 4.
- **Durchschnittswerte:** Sie reichen ungefähr von 2,63 (`E1`) bis 4,03 (`A4`).
- **Auffälligkeit:** Das Minimum `0` bei jeder Frage stammt aus der bereits identifizierten ungültigen Zeile.
- **Bereinigung:** Die Übersicht zeigt noch die unveränderten Rohdaten.

In [25]:
# Häufigkeit aller gender-Werte anzeigen, auch fehlende Angaben
df["gender"].value_counts(dropna=False)

gender
Female    11985
Male       7608
Other       102
NaN          24
Name: count, dtype: int64

### Kategorien in `gender`

- **Female:** 11.985 Einträge
- **Male:** 7.608 Einträge
- **Other:** 102 Einträge
- **Fehlend (`NaN`):** 24 Einträge
- **Abgleich mit dem Codebook:** Keine unerwartete Kategorie gefunden.
- **Bereinigung:** Umgang mit den fehlenden Angaben noch offen.

In [26]:
# Häufigkeit aller hand-Werte anzeigen, auch fehlende Angaben
df["hand"].value_counts(dropna=False)

hand
Right    17424
Left      1724
Both       471
NaN        100
Name: count, dtype: int64

### Kategorien in `hand`

- **Right:** 17.424 Einträge
- **Left:** 1.724 Einträge
- **Both:** 471 Einträge
- **Fehlend (`NaN`):** 100 Einträge
- **Abgleich mit dem Codebook:** Keine unerwartete Kategorie gefunden.
- **Bereinigung:** Umgang mit den fehlenden Angaben noch offen.

Damit haben wir die bisher geplanten Rohdaten-Prüfungen abgeschlossen.

## Punkt 1 abgeschlossen: Datenqualität geprüft

Die relevanten Qualitätsprobleme sind identifiziert. Bis zu diesem Punkt wurden die Rohdaten nur untersucht und noch nicht verändert.

# 2. Daten bereinigen

Jetzt werden die zuvor identifizierten Probleme in der Arbeitskopie `df_clean` behandelt. Die ursprüngliche Tabelle `df` und die Rohdatei `data/data.csv` bleiben unverändert.

In [27]:
# N1: I get stressed out easily.
# Die einzige Zeile mit N1 = 0 hat in allen 19 Fragen den ungültigen Wert 0
df_clean = df[df["N1"] != 0].copy()

# Kontrollieren, wie viele Zeilen vorher und nachher vorhanden sind
print("Vorher:", len(df), "Nachher:", len(df_clean))

Vorher: 19719 Nachher: 19718


### Bereinigung der Fragebewertungen

- **Problem:** Eine Zeile enthielt bei allen 19 Fragen den ungültigen Wert `0`.
- **Entscheidung:** Diese Zeile wurde aus der bereinigten Arbeitskopie entfernt, da der gesamte Fragebogen nicht sinnvoll ergänzt werden kann.
- **Kontrolle:** Vorher 19.719 Zeilen, danach 19.718 Zeilen.
- **Rohdaten:** `df` und `data.csv` bleiben unverändert.

In [28]:
# Auffällige Alterswerte in der bereinigten Arbeitskopie zählen
(df_clean["age"] > 120).sum()

np.int64(82)

### Auffällige Alterswerte nach der ersten Bereinigung

- **Ergebnis:** In `df_clean` liegen weiterhin 82 Altersangaben über 120 Jahren.
- **Einordnung:** Die entfernte Zeile mit 19 Null-Antworten war keiner dieser 82 Fälle.
- **Stand:** Alterswerte wurden noch nicht verändert.

In [29]:
# Altersangaben über 120 Jahren in der Arbeitskopie als fehlend markieren
df_clean.loc[df_clean["age"] > 120, "age"] = pd.NA

# Kontrollieren: keine Werte über 120 mehr, dafür 82 fehlende Altersangaben
print("Über 120:", (df_clean["age"] > 120).sum())
print("Alter fehlt:", df_clean["age"].isna().sum())

Über 120: 0
Alter fehlt: 82


### Bereinigung auffälliger Alterswerte

- **Entscheidung:** Altersangaben über 120 Jahre wurden in `df_clean` als fehlend markiert.
- **Kontrolle:** 0 Alterswerte über 120; 82 fehlende Altersangaben.
- **Erhalten:** Die 82 Zeilen und ihre übrigen Angaben bleiben im Datensatz.
- **Später:** Die Preprocessing-Pipeline behandelt die fehlenden Alterswerte.
- **Rohdaten:** `df` und `data.csv` bleiben unverändert.

## Punkt 2 abgeschlossen: Daten bereinigt

Die vollständig ungültige Fragebogenzeile wurde entfernt und die unplausiblen Alterswerte wurden als fehlend markiert. Weitere fehlende Werte bleiben bewusst für die spätere Preprocessing-Pipeline erhalten.

# 3. Bereinigung kontrollieren und Daten speichern

Zum Abschluss prüfen wir den Zustand der vollständig bereinigten Arbeitskopie, speichern sie als neue Datei und kontrollieren, ob die gespeicherte Datei die erwartete Form besitzt.

In [30]:
# Fehlende Werte in den drei demografischen Spalten von df_clean zählen
df_clean[["age", "gender", "hand"]].isna().sum()

age        82
gender     24
hand      100
dtype: int64

### Verbleibende fehlende Werte

- **`age`:** 82 fehlende Werte; die ungültigen Altersangaben wurden als fehlend markiert.
- **`gender`:** 24 fehlende Angaben.
- **`hand`:** 100 fehlende Angaben.
- **Stand:** Diese Zeilen bleiben erhalten; die Lücken wurden noch nicht aufgefüllt.

In [31]:
# Bereinigte Arbeitskopie als neue CSV speichern; data.csv bleibt unverändert
df_clean.to_csv("data/data_processed.csv", index=False)

# Neue Datei testweise einlesen und Zeilen sowie Spalten kontrollieren
pd.read_csv("data/data_processed.csv").shape

(19718, 23)

### Bereinigten Datensatz gespeichert

- **Datei:** `data/data_processed.csv`
- **Kontrolle:** 19.718 Zeilen und 23 Spalten
- **Bereinigung:** Eine Zeile mit 19 ungültigen Frageantworten entfernt; 82 ungültige Alterswerte als fehlend markiert.
- **Bewusst noch fehlend:** 82 Werte in `age`, 24 in `gender` und 100 in `hand`. Diese behandelt später die Preprocessing-Pipeline.
- **Rohdatei:** `data/data.csv` blieb unverändert.

## Punkt 3 abgeschlossen: Bereinigung kontrolliert und Daten gespeichert

Die bereinigte Datei `data/data_processed.csv` wurde erfolgreich gespeichert und überprüft. Damit ist die EDA einschließlich Datenprüfung, Bereinigung und Abschlusskontrolle abgeschlossen.